### Fetching PDF form 990 from propublica's API


In [ ]:
import json
import pandas as pd

# Load the JSON file content
file_path = 'filings_results.json'
with open(file_path, 'r') as file:
    filings_data = json.load(file)

# Define the columns for the financial data
columns = [
    "EIN", "Year", "Total Revenue", "Total Functional Expenses",
    "Grants and Contributions", "Other Revenue", "Salaries and Wages", "Net Assets at End","Total Grants Proportion"
]

# Extract relevant data into a list of rows for the DataFrame
rows = []
for ein, years in filings_data.items():
    for year, data in years.items():
        rows.append([
            ein,
            year,
            data.get("totrevenue"),
            data.get("totfuncexpns"),
            data.get("totcntrbgfts"),
            
            data.get("othrsalwages"),
            data.get("totnetassetend"),
        ])

# Create a DataFrame
financial_data = pd.DataFrame(rows, columns=columns)

financial_data

In [ ]:
## THIS WORKS
# Extract grant details and income breakdown for analysis
grant_columns = [
    "EIN", "Year", "Grant Amount", "Grant Description", "Other Revenue",
    "Total Revenue", "Total Grants Proportion"
]

grant_data = []

# Process the data to extract grant-related details and categorize
for ein, years in filings_data.items():
    for year, data in years.items():
        total_revenue = data.get("totrevenue", 0)
        grant_amount = data.get("totcntrbgfts", 0)
        other_revenue = total_revenue - grant_amount if total_revenue and grant_amount else 0

        # Simulate grant description for illustrative purposes (data may need API extension)
        grant_description = data.get("grant_description", "Unspecified")

        # Calculate grant proportion if revenue is non-zero
        grant_proportion = (grant_amount / total_revenue) * 100 if total_revenue else 0

        grant_data.append([
            ein, year, grant_amount, grant_description, other_revenue,
            total_revenue, grant_proportion
        ])

# Create a DataFrame for grant data
grant_df = pd.DataFrame(grant_data, columns=grant_columns)

# Display the DataFrame to the user for analysis and planning
grant_df

In [ ]:
import os
from pdf2image import convert_from_path
import pytesseract
from pytesseract import Output
from PIL import Image
from PyPDF2 import PdfWriter

# Path to Tesseract executable (adjust this for your system)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

# Input and output directories
input_dir = "path_to_your_pdfs"  # Folder containing scanned PDFs
output_dir = "path_to_output_pdfs"  # Folder to save searchable PDFs

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

def convert_to_searchable_pdf(pdf_path, output_path):
    # Convert PDF pages to images
    images = convert_from_path(pdf_path)
    
    # Create a new searchable PDF
    pdf_writer = PdfWriter()
    
    for img in images:
        # Perform OCR on each image
        text = pytesseract.image_to_pdf_or_hocr(img, extension='pdf')
        
        # Append the OCR layer to the PDF writer
        pdf_writer.append(text)
    
    # Save the new searchable PDF
    with open(output_path, "wb") as f_out:
        pdf_writer.write(f_out)

# Process all PDFs in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith(".pdf"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, f"searchable_{filename}")
        print(f"Processing {filename}...")
        convert_to_searchable_pdf(input_path, output_path)
        print(f"Saved searchable PDF to {output_path}")


### Returns a list of organizations matching the given search terms.
API document link: https://projects.propublica.org/nonprofits/api

saves: detailed_organiations_results

In [9]:
import requests
import json

def fetch_organization_data(ein):
    """
    Fetch all available data for an organization using its EIN.
    
    :param ein: Employer Identification Number (EIN) of the nonprofit.
    :return: A dictionary containing all available organization data.
    """
    url = f"https://projects.propublica.org/nonprofits/api/v2/organizations/{ein}.json"
    response = requests.get(url)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching data for EIN {ein}: {response.status_code}")
        return {}

def process_organization_data(data):
    """
    Process the organization data to extract key fields.

    :param data: The full organization data as returned by the API.
    :return: A dictionary of the processed data.
    """
    organization = data.get("organization", {})
    filings_with_data = data.get("filings_with_data", [])
    filings_without_data = data.get("filings_without_data", [])
    
    processed_data = {
        "EIN": organization.get("ein"),
        "Name": organization.get("name"),
        "Address": {
            "Street": organization.get("address"),
            "City": organization.get("city"),
            "State": organization.get("state"),
            "Zipcode": organization.get("zipcode"),
        },
        "Ruling Date": organization.get("ruling_date"),
        "Exemption Number": organization.get("exemption_number"),
        "Tax Period": organization.get("tax_period"),
        "Assets": organization.get("asset_amount"),
        "Income": organization.get("income_amount"),
        "Revenue": organization.get("revenue_amount"),
        "NTEE Code": organization.get("ntee_code"),
        "Latest Object ID": organization.get("latest_object_id"),
        "Filings with Data": filings_with_data,
        "Filings without Data": filings_without_data,
    }
    return processed_data

def save_results(results, filename="detailed_organization_results.json"):
    """
    Save the results to a JSON file.

    :param results: The dictionary of results to save.
    :param filename: Name of the file to save the results in.
    """
    with open(filename, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {filename}")

def main():
    print("Fetching all data for specific EINs...")

    eins = [
        "91-1767292", "33-0647946", "84-4780735", "45-4329874", "83-3305529",
        "33-0587567", "94-2745941", "33-0145660", "94-2665367", "20-1832617",
        "47-5161428", "94-2216915", "23-7213237", "85-3432174", "33-0974992",
        "94-1693226", "22-3902362", "95-6377791", "33-0537412", "45-3042628",
        "95-2496099", "45-2639830", "95-3273023", "68-0120240", "95-2566791",
        "77-0565183", "82-4594246", "83-1159078", "01-0777856", "38-3891081",
        "94-2951488", "94-2788588", "88-1650309", "43-2050242", "30-0358349",
        "46-5112972"
    ]

    results = {}

    for ein in eins:
        print(f"Fetching data for EIN: {ein}")
        data = fetch_organization_data(ein)
        if data:
            results[ein] = process_organization_data(data)
        else:
            print(f"No data found for EIN {ein}.")

    save_results(results)

if __name__ == "__main__":
    main()


Fetching all data for specific EINs...
Fetching data for EIN: 91-1767292
Fetching data for EIN: 33-0647946
Fetching data for EIN: 84-4780735
Fetching data for EIN: 45-4329874
Fetching data for EIN: 83-3305529
Fetching data for EIN: 33-0587567
Fetching data for EIN: 94-2745941
Fetching data for EIN: 33-0145660
Fetching data for EIN: 94-2665367
Fetching data for EIN: 20-1832617
Fetching data for EIN: 47-5161428
Fetching data for EIN: 94-2216915
Fetching data for EIN: 23-7213237
Fetching data for EIN: 85-3432174
Fetching data for EIN: 33-0974992
Fetching data for EIN: 94-1693226
Fetching data for EIN: 22-3902362
Fetching data for EIN: 95-6377791
Fetching data for EIN: 33-0537412
Fetching data for EIN: 45-3042628
Fetching data for EIN: 95-2496099
Fetching data for EIN: 45-2639830
Fetching data for EIN: 95-3273023
Fetching data for EIN: 68-0120240
Fetching data for EIN: 95-2566791
Fetching data for EIN: 77-0565183
Fetching data for EIN: 82-4594246
Fetching data for EIN: 83-1159078
Fetching 

In [ ]:
# Save to CSV
output_file = "nonprofit_full_data.csv"
df.to_csv(output_file, index=False)

print(f"Data saved to {output_file}")